In [1]:
from pathlib import Path
import json
import sys

from dataclasses import dataclass
from collections import Counter, defaultdict
import copy

from joblib import Parallel, delayed
import os
import time

import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance


project_dir = Path.cwd()

raw_dir = project_dir / "data" / "raw"
output_dir = project_dir / "outputs" / "odds_40"

results_path = raw_dir / "results.csv"
rankings_path = raw_dir / "fifa_consolidado.csv"
schedule_path = raw_dir / "world_cup_2026_schedule.csv"
odds_json_path = raw_dir / "odds_fase_grupos_copa.json"
odds_csv_path = raw_dir / "odds.csv"

output_dir.mkdir(parents=True, exist_ok=True)

print(project_dir)
print(raw_dir.exists())

@dataclass
class TournamentSimulationRounds:
    champion: str
    runner_up: str
    third_place: str
    fourth_place: str
    finalists: list[str]
    semifinalists: list[str]
    quarterfinalists: list[str]
    round_of_16_teams: list[str]
    round_of_32_teams: list[str]
    group_advancers: list[str]
    playoff_qualifiers: list[str]

d:\windows\Documents\extras\previsao_copa_26
True


In [2]:
for path in [results_path, rankings_path, schedule_path, odds_json_path]:
    print(path, path.exists())

d:\windows\Documents\extras\previsao_copa_26\data\raw\results.csv True
d:\windows\Documents\extras\previsao_copa_26\data\raw\fifa_consolidado.csv True
d:\windows\Documents\extras\previsao_copa_26\data\raw\world_cup_2026_schedule.csv True
d:\windows\Documents\extras\previsao_copa_26\data\raw\odds_fase_grupos_copa.json True


In [3]:
TEAM_NAME_MAP = {
    "USA": "United States",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Curaçao": "Curacao",
    "Côte d'Ivoire": "Ivory Coast",
    "Czech Republic": "Czechia",
    "Türkiye": "Turkey",
}


def clean_team_name(team_name):
    team_name = str(team_name).strip()
    return TEAM_NAME_MAP.get(team_name, team_name)


def extract_h2h_prices(event):
    home_team = clean_team_name(event["home_team"])
    away_team = clean_team_name(event["away_team"])

    home_prices = []
    draw_prices = []
    away_prices = []

    for bookmaker in event.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market.get("key") != "h2h":
                continue

            prices_by_name = {
                clean_team_name(outcome["name"]): outcome["price"]
                for outcome in market.get("outcomes", [])
            }

            if home_team in prices_by_name:
                home_prices.append(float(prices_by_name[home_team]))

            if away_team in prices_by_name:
                away_prices.append(float(prices_by_name[away_team]))

            if "Draw" in prices_by_name:
                draw_prices.append(float(prices_by_name["Draw"]))

    if not home_prices or not draw_prices or not away_prices:
        return None

    return {
        "event_date": pd.to_datetime(event["commence_time"]).date(),
        "event_home_team": home_team,
        "event_away_team": away_team,
        "event_home_win_odds": float(np.median(home_prices)),
        "event_draw_odds": float(np.median(draw_prices)),
        "event_away_win_odds": float(np.median(away_prices)),
        "n_bookmakers": len(home_prices),
    }


with odds_json_path.open("r", encoding="utf-8") as file:
    odds_events = json.load(file)

odds_rows = []

for event in odds_events:
    extracted_prices = extract_h2h_prices(event)

    if extracted_prices is not None:
        odds_rows.append(extracted_prices)

odds_data = pd.DataFrame(odds_rows)

print(odds_data.shape)
odds_data.head()

(72, 7)


,event_date,event_home_team,event_away_team,event_home_win_odds,event_draw_odds,event_away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-12,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-13,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [4]:
schedule_data = pd.read_csv(schedule_path)

schedule_data["home_team"] = schedule_data["home_team"].map(clean_team_name)
schedule_data["away_team"] = schedule_data["away_team"].map(clean_team_name)

aligned_rows = []

for _, odds_row in odds_data.iterrows():
    event_home_team = odds_row["event_home_team"]
    event_away_team = odds_row["event_away_team"]

    same_order = schedule_data[
        (schedule_data["home_team"] == event_home_team)
        & (schedule_data["away_team"] == event_away_team)
    ]

    reverse_order = schedule_data[
        (schedule_data["home_team"] == event_away_team)
        & (schedule_data["away_team"] == event_home_team)
    ]

    if not same_order.empty:
        schedule_row = same_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_home_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_away_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    elif not reverse_order.empty:
        schedule_row = reverse_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_away_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_home_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    else:
        print("Não encontrei na tabela:", event_home_team, "x", event_away_team)

odds_aligned_data = pd.DataFrame(aligned_rows)

odds_aligned_data.to_csv(odds_csv_path, index=False)

print(odds_aligned_data.shape)
odds_aligned_data.head()

(72, 7)


,match_date,home_team,away_team,home_win_odds,draw_odds,away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-11,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-12,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [5]:
import poison_26 as path_a

In [6]:
historical_matches_raw = pd.read_csv(results_path)
rankings_raw = pd.read_csv(rankings_path)
schedule_raw = pd.read_csv(schedule_path)
odds_raw = pd.read_csv(odds_csv_path)

historical_matches_raw = historical_matches_raw.copy()

score_columns = ["home_score", "away_score"]

historical_matches_raw["date"] = pd.to_datetime(
    historical_matches_raw["date"],
    errors="coerce",
)

historical_matches_raw["home_score"] = pd.to_numeric(
    historical_matches_raw["home_score"],
    errors="coerce",
)

historical_matches_raw["away_score"] = pd.to_numeric(
    historical_matches_raw["away_score"],
    errors="coerce",
)

missing_score_data = historical_matches_raw[
    historical_matches_raw[score_columns].isna().any(axis=1)
].copy()

print("Jogos sem placar:", missing_score_data.shape[0])

display(
    missing_score_data[
        ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]
    ].tail(20)
)

historical_matches_clean = historical_matches_raw.dropna(
    subset=["date", "home_score", "away_score"]
).copy()

historical_matches_clean = historical_matches_clean[
    historical_matches_clean["date"] < pd.Timestamp("2026-06-11")
].copy()

historical_matches_clean["home_score"] = historical_matches_clean["home_score"].astype(int)
historical_matches_clean["away_score"] = historical_matches_clean["away_score"].astype(int)

historical_matches = path_a.standardize_matches_columns(historical_matches_clean)
schedule_data = path_a.standardize_schedule_columns(schedule_raw)

ranking_lookup = path_a.RankingLookup(rankings_raw)
odds_lookup = path_a.OddsLookup(odds_raw)

print("Histórico bruto:", historical_matches_raw.shape)
print("Histórico limpo:", historical_matches.shape)
print("Tabela Copa:", schedule_data.shape)
print("Odds:", odds_raw.shape)

Jogos sem placar: 72


,date,home_team,away_team,home_score,away_score,tournament
49452,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup
49453,2026-06-24,Morocco,Haiti,NaN,NaN,FIFA World Cup
49454,2026-06-25,United States,Turkey,NaN,NaN,FIFA World Cup
49455,2026-06-25,Paraguay,Australia,NaN,NaN,FIFA World Cup
49456,2026-06-25,Curaçao,Ivory Coast,NaN,NaN,FIFA World Cup
49457,2026-06-25,Ecuador,Germany,NaN,NaN,FIFA World Cup
49458,2026-06-25,Japan,Sweden,NaN,NaN,FIFA World Cup
49459,2026-06-25,Tunisia,Netherlands,NaN,NaN,FIFA World Cup
49460,2026-06-26,Egypt,Iran,NaN,NaN,FIFA World Cup
49461,2026-06-26,New Zealand,Belgium,NaN,NaN,FIFA World Cup


Histórico bruto: (49472, 9)
Histórico limpo: (49400, 9)
Tabela Copa: (72, 8)
Odds: (72, 7)


In [7]:
market_features = [
    "market_home_prob",
    "market_draw_prob",
    "market_away_prob",
]

path_a.FEATURE_COLUMNS = [
    feature
    for feature in path_a.FEATURE_COLUMNS
    if feature not in market_features
]

odds_lookup_for_training = path_a.OddsLookup(None)

features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup_for_training,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(features_data.shape)
print(path_a.FEATURE_COLUMNS)

(49400, 23)
['elo_diff', 'attack_diff', 'defense_diff', 'form_diff', 'home_rank', 'away_rank', 'rank_diff', 'home_total_points', 'away_total_points', 'fifa_points_diff', 'neutral', 'is_non_neutral', 'tournament_weight', 'home_attack', 'away_attack', 'home_defense', 'away_defense', 'home_form', 'away_form', 'home_decay', 'away_decay', 'home_matches_played', 'away_matches_played']


In [8]:
features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(home_goals.shape)
print(away_goals.shape)

(49400,)
(49400,)


In [9]:
FEATURE_COLUMNS_USED = list(path_a.FEATURE_COLUMNS)

print(len(FEATURE_COLUMNS_USED))
print(FEATURE_COLUMNS_USED)

23
['elo_diff', 'attack_diff', 'defense_diff', 'form_diff', 'home_rank', 'away_rank', 'rank_diff', 'home_total_points', 'away_total_points', 'fifa_points_diff', 'neutral', 'is_non_neutral', 'tournament_weight', 'home_attack', 'away_attack', 'home_defense', 'away_defense', 'home_form', 'away_form', 'home_decay', 'away_decay', 'home_matches_played', 'away_matches_played']


In [10]:
def split_simulations(total_simulations, n_chunks):
    base_size = total_simulations // n_chunks
    remainder = total_simulations % n_chunks

    chunk_sizes = []

    for chunk_index in range(n_chunks):
        chunk_size = base_size + (1 if chunk_index < remainder else 0)

        if chunk_size > 0:
            chunk_sizes.append(chunk_size)

    return chunk_sizes

In [11]:
def extract_poisson_weights(model, model_name):
    scaler = model.named_steps["scaler"]
    poisson = model.named_steps["poisson"]

    standardized_coefficients = pd.Series(
        poisson.coef_,
        index=path_a.FEATURE_COLUMNS,
        name="standardized_coefficient",
    )

    raw_scale_coefficients = standardized_coefficients / scaler.scale_

    weights_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "standardized_coefficient": standardized_coefficients.values,
            "raw_scale_coefficient": raw_scale_coefficients.values,
            "abs_standardized_coefficient": standardized_coefficients.abs().values,
        }
    )

    weights_data["model"] = model_name

    return weights_data.sort_values(
        "abs_standardized_coefficient",
        ascending=False,
    ).reset_index(drop=True)


home_weights = extract_poisson_weights(home_model, "home_goals")
away_weights = extract_poisson_weights(away_model, "away_goals")

display(home_weights.head(30))
display(away_weights.head(30))

,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,0.358338,0.001530,0.358338,home_goals
1,away_defense,0.112142,0.151153,0.112142,home_goals
2,away_matches_played,-0.106291,-0.000476,0.106291,home_goals
3,form_diff,-0.105873,-0.505633,0.105873,home_goals
4,home_attack,0.093331,0.154615,0.093331,home_goals
5,defense_diff,0.085472,0.101656,0.085472,home_goals
6,away_form,0.079743,0.499623,0.079743,home_goals
7,attack_diff,0.064021,0.089894,0.064021,home_goals
8,home_form,-0.058985,-0.368527,0.058985,home_goals
9,away_decay,-0.045134,-0.282171,0.045134,home_goals


,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,-0.364258,-0.001555,0.364258,away_goals
1,home_defense,0.129155,0.184120,0.129155,away_goals
2,away_attack,0.116105,0.194344,0.116105,away_goals
3,form_diff,0.099475,0.475079,0.099475,away_goals
4,defense_diff,-0.091278,-0.108562,0.091278,away_goals
5,away_form,-0.078473,-0.491669,0.078473,away_goals
6,home_matches_played,-0.059252,-0.000255,0.059252,away_goals
7,home_attack,0.054545,0.090361,0.054545,away_goals
8,home_form,0.051881,0.324143,0.051881,away_goals
9,attack_diff,-0.051164,-0.071841,0.051164,away_goals


In [12]:
def calculate_permutation_importance(model, features_data, target, model_name):
    result = permutation_importance(
        model,
        features_data[path_a.FEATURE_COLUMNS],
        target,
        n_repeats=10,
        random_state=42,
        scoring="neg_mean_absolute_error",
    )

    importance_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
            "model": model_name,
        }
    )

    return importance_data.sort_values(
        "importance_mean",
        ascending=False,
    ).reset_index(drop=True)


home_importance = calculate_permutation_importance(
    home_model,
    features_data,
    home_goals,
    "home_goals",
)

away_importance = calculate_permutation_importance(
    away_model,
    features_data,
    away_goals,
    "away_goals",
)

display(home_importance.head(30))
display(away_importance.head(30))

,feature,importance_mean,importance_std,model
0,elo_diff,0.203672,0.001251,home_goals
1,form_diff,0.040188,0.001120,home_goals
2,away_defense,0.024466,0.000755,home_goals
3,away_form,0.023181,0.000848,home_goals
4,home_attack,0.013217,0.000526,home_goals
5,away_matches_played,0.010969,0.000935,home_goals
6,home_form,0.010057,0.000513,home_goals
7,defense_diff,0.009850,0.000505,home_goals
8,attack_diff,0.002742,0.000547,home_goals
9,away_decay,0.002137,0.000417,home_goals


,feature,importance_mean,importance_std,model
0,elo_diff,0.119668,0.002107,away_goals
1,form_diff,0.023555,0.000554,away_goals
2,home_defense,0.016762,0.000632,away_goals
3,away_form,0.011646,0.000408,away_goals
4,away_attack,0.009545,0.000599,away_goals
5,home_form,0.007371,0.000335,away_goals
6,home_attack,0.005591,0.000379,away_goals
7,defense_diff,0.004690,0.000456,away_goals
8,is_non_neutral,0.002359,0.000343,away_goals
9,neutral,0.002359,0.000343,away_goals


In [13]:
MARKET_WEIGHT = 0.4
MAX_GOALS = 10


def calculate_poisson_probabilities(goal_rate, max_goals=MAX_GOALS):
    probabilities = np.zeros(max_goals + 1)
    probabilities[0] = np.exp(-goal_rate)

    for goals in range(1, max_goals + 1):
        probabilities[goals] = probabilities[goals - 1] * goal_rate / goals

    probabilities = probabilities / probabilities.sum()

    return probabilities


def calculate_poisson_outcome_probabilities(
    home_lambda,
    away_lambda,
    max_goals=MAX_GOALS,
):
    home_probabilities = calculate_poisson_probabilities(home_lambda, max_goals)
    away_probabilities = calculate_poisson_probabilities(away_lambda, max_goals)

    score_matrix = np.outer(home_probabilities, away_probabilities)

    home_win_probability = np.tril(score_matrix, k=-1).sum()
    draw_probability = np.trace(score_matrix)
    away_win_probability = np.triu(score_matrix, k=1).sum()

    total_probability = (
        home_win_probability
        + draw_probability
        + away_win_probability
    )

    return {
        "home_win": home_win_probability / total_probability,
        "draw": draw_probability / total_probability,
        "away_win": away_win_probability / total_probability,
    }


def normalize_probabilities(probabilities):
    total_probability = sum(probabilities.values())

    if total_probability <= 0:
        raise ValueError("A soma das probabilidades precisa ser positiva.")

    return {
        key: value / total_probability
        for key, value in probabilities.items()
    }


def market_probabilities_to_outcome_probabilities(market_probabilities):
    return normalize_probabilities(
        {
            "home_win": market_probabilities["market_home_prob"],
            "draw": market_probabilities["market_draw_prob"],
            "away_win": market_probabilities["market_away_prob"],
        }
    )


def has_real_market_probabilities(market_probabilities):
    default_probabilities = path_a.DEFAULT_MARKET_PROBABILITIES

    return any(
        abs(market_probabilities[key] - default_probabilities[key]) > 1e-10
        for key in default_probabilities
    )


def blend_outcome_probabilities(
    model_probabilities,
    market_probabilities,
    market_weight=MARKET_WEIGHT,
):
    if not has_real_market_probabilities(market_probabilities):
        return normalize_probabilities(model_probabilities)

    market_outcome_probabilities = market_probabilities_to_outcome_probabilities(
        market_probabilities
    )

    blended_probabilities = {
        outcome: (
            (1.0 - market_weight) * model_probabilities[outcome]
            + market_weight * market_outcome_probabilities[outcome]
        )
        for outcome in ["home_win", "draw", "away_win"]
    }

    return normalize_probabilities(blended_probabilities)

In [14]:
def get_score_outcome(home_goals, away_goals):
    if home_goals > away_goals:
        return "home_win"

    if home_goals == away_goals:
        return "draw"

    return "away_win"


def sample_score_from_blended_probabilities(
    home_lambda,
    away_lambda,
    target_probabilities,
    random_generator,
    max_goals=MAX_GOALS,
):
    home_probabilities = calculate_poisson_probabilities(home_lambda, max_goals)
    away_probabilities = calculate_poisson_probabilities(away_lambda, max_goals)

    score_matrix = np.outer(home_probabilities, away_probabilities)

    model_outcome_probabilities = {
        "home_win": 0.0,
        "draw": 0.0,
        "away_win": 0.0,
    }

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            outcome = get_score_outcome(home_goals, away_goals)
            model_outcome_probabilities[outcome] += score_matrix[
                home_goals,
                away_goals,
            ]

    adjusted_score_matrix = score_matrix.copy()

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            outcome = get_score_outcome(home_goals, away_goals)
            model_probability = model_outcome_probabilities[outcome]

            if model_probability > 0:
                adjustment_factor = (
                    target_probabilities[outcome] / model_probability
                )
                adjusted_score_matrix[home_goals, away_goals] *= adjustment_factor

    adjusted_score_matrix = adjusted_score_matrix / adjusted_score_matrix.sum()

    flat_probabilities = adjusted_score_matrix.ravel()
    selected_index = random_generator.choice(
        len(flat_probabilities),
        p=flat_probabilities,
    )

    home_goals, away_goals = np.unravel_index(
        selected_index,
        adjusted_score_matrix.shape,
    )

    return int(home_goals), int(away_goals)

In [15]:
def simulate_match_with_market_blend(
    home_team,
    away_team,
    match_date,
    neutral,
    tournament,
    stage,
    group,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
    knockout=False,
):
    market_probabilities = odds_lookup.get(
        match_date,
        home_team,
        away_team,
    )

    feature_row = path_a.build_feature_row(
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        neutral=neutral,
        tournament=tournament,
        states=states,
        ranking_lookup=ranking_lookup,
        market_probabilities=market_probabilities,
    )

    home_lambda, away_lambda = path_a.predict_expected_goals(
        home_model,
        away_model,
        feature_row,
    )

    model_probabilities = calculate_poisson_outcome_probabilities(
        home_lambda,
        away_lambda,
    )

    target_probabilities = blend_outcome_probabilities(
        model_probabilities=model_probabilities,
        market_probabilities=market_probabilities,
        market_weight=MARKET_WEIGHT,
    )

    home_goals, away_goals = sample_score_from_blended_probabilities(
        home_lambda=home_lambda,
        away_lambda=away_lambda,
        target_probabilities=target_probabilities,
        random_generator=random_generator,
    )

    winner = path_a.infer_winner_from_score(
        home_team,
        away_team,
        home_goals,
        away_goals,
    )

    home_result_override = None

    if knockout and winner is None:
        home_advancement_probability = (
            target_probabilities["home_win"]
            / (
                target_probabilities["home_win"]
                + target_probabilities["away_win"]
            )
        )

        home_advancement_probability = float(
            np.clip(home_advancement_probability, 0.35, 0.65)
        )

        if random_generator.random() < home_advancement_probability:
            winner = home_team
            home_result_override = 0.55
        else:
            winner = away_team
            home_result_override = 0.45

    path_a.update_team_states(
        home_team=home_team,
        away_team=away_team,
        home_goals=home_goals,
        away_goals=away_goals,
        match_date=match_date,
        tournament=tournament,
        states=states,
        home_result_override=home_result_override,
    )

    return path_a.MatchSimulation(
        home_team=home_team,
        away_team=away_team,
        home_goals=home_goals,
        away_goals=away_goals,
        winner=winner,
        stage=stage,
        group=group,
    )


path_a.simulate_match = simulate_match_with_market_blend

In [16]:
ROUND_OF_32_RULES = {
    "M73": ("2A", "2B"),
    "M74": ("1E", "3"),
    "M75": ("1F", "2C"),
    "M76": ("1C", "2F"),
    "M77": ("1I", "3"),
    "M78": ("2E", "2I"),
    "M79": ("1A", "3"),
    "M80": ("1L", "3"),
    "M81": ("1D", "3"),
    "M82": ("1G", "3"),
    "M83": ("2K", "2L"),
    "M84": ("1H", "2J"),
    "M85": ("1B", "3"),
    "M86": ("1J", "2H"),
    "M87": ("1K", "3"),
    "M88": ("2D", "2G"),
}

THIRD_PLACE_SLOT_GROUPS = {
    "M74": {"A", "B", "C", "D", "F"},
    "M77": {"C", "D", "F", "G", "H"},
    "M79": {"C", "E", "F", "H", "I"},
    "M80": {"E", "H", "I", "J", "K"},
    "M81": {"B", "E", "F", "I", "J"},
    "M82": {"A", "E", "H", "I", "J"},
    "M85": {"E", "F", "G", "I", "J"},
    "M87": {"D", "E", "I", "J", "L"},
}

KNOCKOUT_RULES = {
    "M89": ("W73", "W75"),
    "M90": ("W74", "W77"),
    "M91": ("W76", "W78"),
    "M92": ("W79", "W80"),
    "M93": ("W83", "W84"),
    "M94": ("W81", "W82"),
    "M95": ("W86", "W88"),
    "M96": ("W85", "W87"),
    "M97": ("W89", "W90"),
    "M98": ("W93", "W94"),
    "M99": ("W91", "W92"),
    "M100": ("W95", "W96"),
    "M101": ("W97", "W98"),
    "M102": ("W99", "W100"),
    "M103": ("L101", "L102"),
    "M104": ("W101", "W102"),
}

In [17]:
def simulate_group_stage_for_official_bracket(
    group_matches,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    group_tables = defaultdict(dict)

    for _, match_row in group_matches.iterrows():
        group_name = str(match_row["group"])
        home_team = match_row["home_team"]
        away_team = match_row["away_team"]

        path_a.ensure_group_entry(group_tables, group_name, home_team)
        path_a.ensure_group_entry(group_tables, group_name, away_team)

        match = path_a.simulate_match(
            home_team=home_team,
            away_team=away_team,
            match_date=pd.Timestamp(match_row["match_date"]),
            neutral=int(match_row["neutral"]),
            tournament=match_row["tournament"],
            stage=match_row["stage"],
            group=group_name,
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
            knockout=False,
        )

        path_a.update_group_table(
            group_tables[group_name],
            match.home_team,
            match.away_team,
            match.home_goals,
            match.away_goals,
        )

    ranked_groups = {
        group_name: path_a.rank_group_table(table_entries, random_generator)
        for group_name, table_entries in group_tables.items()
    }

    group_winners = {}
    group_runners_up = {}
    third_places = []
    group_advancers = []

    for group_name, ranked_table in ranked_groups.items():
        if len(ranked_table) < 4:
            raise ValueError(f"Grupo {group_name} tem menos de 4 times.")

        winner = ranked_table[0]["team"]
        runner_up = ranked_table[1]["team"]

        group_winners[group_name] = winner
        group_runners_up[group_name] = runner_up

        group_advancers.extend([winner, runner_up])

        third_entry = ranked_table[2].copy()
        third_entry["group"] = group_name
        third_places.append(third_entry)

    best_third_places = path_a.rank_third_places(
        third_places,
        random_generator,
    )[:8]

    group_advancers.extend([entry["team"] for entry in best_third_places])

    return {
        "ranked_groups": ranked_groups,
        "group_winners": group_winners,
        "group_runners_up": group_runners_up,
        "best_third_places": best_third_places,
        "group_advancers": group_advancers,
    }

In [18]:
def resolve_third_place_slots(best_third_places):
    available_groups = {
        str(entry["group"]): entry["team"]
        for entry in best_third_places
    }

    group_priority = {
        str(entry["group"]): position
        for position, entry in enumerate(best_third_places)
    }

    slot_order = sorted(
        THIRD_PLACE_SLOT_GROUPS,
        key=lambda slot: len(
            THIRD_PLACE_SLOT_GROUPS[slot].intersection(available_groups)
        ),
    )

    def backtrack(slot_index, remaining_groups, current_assignment):
        if slot_index == len(slot_order):
            return current_assignment

        slot = slot_order[slot_index]

        eligible_groups = [
            group
            for group in remaining_groups
            if group in THIRD_PLACE_SLOT_GROUPS[slot]
        ]

        eligible_groups = sorted(
            eligible_groups,
            key=lambda group: group_priority[group],
        )

        for group in eligible_groups:
            next_assignment = dict(current_assignment)
            next_assignment[slot] = remaining_groups[group]

            next_remaining_groups = dict(remaining_groups)
            del next_remaining_groups[group]

            solved_assignment = backtrack(
                slot_index + 1,
                next_remaining_groups,
                next_assignment,
            )

            if solved_assignment is not None:
                return solved_assignment

        return None

    assignment = backtrack(
        slot_index=0,
        remaining_groups=available_groups,
        current_assignment={},
    )

    if assignment is None:
        raise RuntimeError(
            "Não foi possível encaixar os melhores terceiros nos slots oficiais."
        )

    return assignment

In [19]:
def resolve_group_position_token(
    token,
    group_winners,
    group_runners_up,
):
    position = token[0]
    group = token[1]

    if position == "1":
        return group_winners[group]

    if position == "2":
        return group_runners_up[group]

    raise ValueError(f"Token inválido: {token}")


def create_official_round_of_32_pairs(
    group_winners,
    group_runners_up,
    best_third_places,
):
    third_place_assignment = resolve_third_place_slots(best_third_places)

    match_pairs = {}

    for match_id, (home_token, away_token) in ROUND_OF_32_RULES.items():
        if home_token == "3":
            home_team = third_place_assignment[match_id]
        else:
            home_team = resolve_group_position_token(
                home_token,
                group_winners,
                group_runners_up,
            )

        if away_token == "3":
            away_team = third_place_assignment[match_id]
        else:
            away_team = resolve_group_position_token(
                away_token,
                group_winners,
                group_runners_up,
            )

        match_pairs[match_id] = (home_team, away_team)

    return match_pairs

In [20]:
def get_loser_from_match(home_team, away_team, winner):
    return away_team if winner == home_team else home_team


def resolve_knockout_rule(rule, match_results):
    result = match_results[f"M{rule[1:]}"]

    if rule.startswith("W"):
        return result.winner

    if rule.startswith("L"):
        return get_loser_from_match(
            result.home_team,
            result.away_team,
            result.winner,
        )

    raise ValueError(f"Regra inválida: {rule}")


def simulate_official_knockout_match(
    match_id,
    home_team,
    away_team,
    stage,
    match_date,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    match = path_a.simulate_match(
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        neutral=1,
        tournament="FIFA World Cup",
        stage=stage,
        group=None,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
        knockout=True,
    )

    if match.winner is None:
        raise RuntimeError(f"{match_id} terminou sem vencedor.")

    return match

In [21]:
def simulate_official_knockout_bracket_with_round_tracking(
    group_winners,
    group_runners_up,
    best_third_places,
    start_date,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    match_results = {}

    round_of_32_pairs = create_official_round_of_32_pairs(
        group_winners=group_winners,
        group_runners_up=group_runners_up,
        best_third_places=best_third_places,
    )

    round_of_32_teams = sorted(
        {team for pair in round_of_32_pairs.values() for team in pair}
    )

    round_of_16_teams = []
    quarterfinalists = []
    semifinalists = []
    finalists = []

    match_stage = {}
    for match_number in range(73, 89):
        match_stage[f"M{match_number}"] = "round_of_32"
    for match_number in range(89, 97):
        match_stage[f"M{match_number}"] = "round_of_16"
    for match_number in range(97, 101):
        match_stage[f"M{match_number}"] = "quarterfinal"
    for match_number in range(101, 103):
        match_stage[f"M{match_number}"] = "semifinal"
    match_stage["M103"] = "third_place"
    match_stage["M104"] = "final"

    for match_number in range(73, 89):
        match_id = f"M{match_number}"
        home_team, away_team = round_of_32_pairs[match_id]

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=match_number - 73),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    round_of_16_teams = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(73, 89)
    )

    for match_number in range(89, 97):
        match_id = f"M{match_number}"
        home_rule, away_rule = KNOCKOUT_RULES[match_id]

        home_team = resolve_knockout_rule(home_rule, match_results)
        away_team = resolve_knockout_rule(away_rule, match_results)

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=20 + match_number - 89),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    quarterfinalists = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(89, 97)
    )

    for match_number in range(97, 101):
        match_id = f"M{match_number}"
        home_rule, away_rule = KNOCKOUT_RULES[match_id]

        home_team = resolve_knockout_rule(home_rule, match_results)
        away_team = resolve_knockout_rule(away_rule, match_results)

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=32 + match_number - 97),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    semifinalists = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(97, 101)
    )

    for match_number in range(101, 103):
        match_id = f"M{match_number}"
        home_rule, away_rule = KNOCKOUT_RULES[match_id]

        home_team = resolve_knockout_rule(home_rule, match_results)
        away_team = resolve_knockout_rule(away_rule, match_results)

        match_results[match_id] = simulate_official_knockout_match(
            match_id=match_id,
            home_team=home_team,
            away_team=away_team,
            stage=match_stage[match_id],
            match_date=start_date + pd.Timedelta(days=42 + match_number - 101),
            states=states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

    finalists = sorted(
        match_results[f"M{match_number}"].winner
        for match_number in range(101, 103)
    )

    third_home = resolve_knockout_rule("L101", match_results)
    third_away = resolve_knockout_rule("L102", match_results)

    match_results["M103"] = simulate_official_knockout_match(
        match_id="M103",
        home_team=third_home,
        away_team=third_away,
        stage="third_place",
        match_date=start_date + pd.Timedelta(days=49),
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    final_home = resolve_knockout_rule("W101", match_results)
    final_away = resolve_knockout_rule("W102", match_results)

    match_results["M104"] = simulate_official_knockout_match(
        match_id="M104",
        home_team=final_home,
        away_team=final_away,
        stage="final",
        match_date=start_date + pd.Timedelta(days=50),
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    final_match = match_results["M104"]
    third_place_match = match_results["M103"]

    champion = final_match.winner
    runner_up = get_loser_from_match(
        final_match.home_team,
        final_match.away_team,
        final_match.winner,
    )

    third_place = third_place_match.winner
    fourth_place = get_loser_from_match(
        third_place_match.home_team,
        third_place_match.away_team,
        third_place_match.winner,
    )

    return {
        "champion": champion,
        "runner_up": runner_up,
        "third_place": third_place,
        "fourth_place": fourth_place,
        "finalists": finalists,
        "semifinalists": semifinalists,
        "quarterfinalists": quarterfinalists,
        "round_of_16_teams": round_of_16_teams,
        "round_of_32_teams": round_of_32_teams,
        "match_results": match_results,
    }

In [22]:
@dataclass
class TournamentSimulationOfficialBracket:
    champion: str
    runner_up: str
    third_place: str
    fourth_place: str
    finalists: list[str]
    semifinalists: list[str]
    quarterfinalists: list[str]
    round_of_16_teams: list[str]
    round_of_32_teams: list[str]
    group_advancers: list[str]
    playoff_qualifiers: list[str]

In [23]:
def run_one_simulation_official_bracket(
    schedule_data,
    playoff_data,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
):
    states = copy.deepcopy(initial_states)

    slots, playoff_qualifiers = path_a.simulate_playoffs(
        playoff_data=playoff_data,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    resolved_schedule = path_a.resolve_schedule_slots(schedule_data, slots)

    group_matches = resolved_schedule[
        resolved_schedule["stage"].str.lower().str.contains("group")
    ].copy()

    if group_matches.empty:
        raise ValueError("A tabela da Copa não tem jogos de fase de grupos.")

    group_result = simulate_group_stage_for_official_bracket(
        group_matches=group_matches,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    knockout_start_date = group_matches["match_date"].max() + pd.Timedelta(days=1)

    knockout_result = simulate_official_knockout_bracket_with_round_tracking(
        group_winners=group_result["group_winners"],
        group_runners_up=group_result["group_runners_up"],
        best_third_places=group_result["best_third_places"],
        start_date=knockout_start_date,
        states=states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        random_generator=random_generator,
    )

    return TournamentSimulationOfficialBracket(
        champion=knockout_result["champion"],
        runner_up=knockout_result["runner_up"],
        third_place=knockout_result["third_place"],
        fourth_place=knockout_result["fourth_place"],
        finalists=knockout_result["finalists"],
        semifinalists=knockout_result["semifinalists"],
        quarterfinalists=knockout_result["quarterfinalists"],
        round_of_16_teams=knockout_result["round_of_16_teams"],
        round_of_32_teams=knockout_result["round_of_32_teams"],
        group_advancers=group_result["group_advancers"],
        playoff_qualifiers=playoff_qualifiers,
    )

In [24]:
def run_simulations_official_bracket(
    schedule_data,
    playoff_data,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    simulations,
    seed,
):
    random_generator = path_a.default_rng(seed)

    champion_counter = Counter()
    runner_up_counter = Counter()
    third_place_counter = Counter()
    fourth_place_counter = Counter()
    finalist_counter = Counter()
    semifinal_counter = Counter()
    quarterfinal_counter = Counter()
    round_of_16_counter = Counter()
    round_of_32_counter = Counter()
    group_advancement_counter = Counter()
    playoff_qualification_counter = Counter()
    champion_given_playoff_counter = Counter()

    for _ in range(simulations):
        simulation = run_one_simulation_official_bracket(
            schedule_data=schedule_data,
            playoff_data=playoff_data,
            initial_states=initial_states,
            ranking_lookup=ranking_lookup,
            odds_lookup=odds_lookup,
            home_model=home_model,
            away_model=away_model,
            random_generator=random_generator,
        )

        champion_counter[simulation.champion] += 1
        runner_up_counter[simulation.runner_up] += 1
        third_place_counter[simulation.third_place] += 1
        fourth_place_counter[simulation.fourth_place] += 1

        for team in simulation.finalists:
            finalist_counter[team] += 1

        for team in simulation.semifinalists:
            semifinal_counter[team] += 1

        for team in simulation.quarterfinalists:
            quarterfinal_counter[team] += 1

        for team in simulation.round_of_16_teams:
            round_of_16_counter[team] += 1

        for team in simulation.round_of_32_teams:
            round_of_32_counter[team] += 1

        for team in simulation.group_advancers:
            group_advancement_counter[team] += 1

        for team in simulation.playoff_qualifiers:
            playoff_qualification_counter[team] += 1

        if simulation.champion in simulation.playoff_qualifiers:
            champion_given_playoff_counter[simulation.champion] += 1

    return {
        "champion": champion_counter,
        "runner_up": runner_up_counter,
        "third_place": third_place_counter,
        "fourth_place": fourth_place_counter,
        "final": finalist_counter,
        "semifinal": semifinal_counter,
        "quarterfinal": quarterfinal_counter,
        "round_of_16": round_of_16_counter,
        "round_of_32": round_of_32_counter,
        "group_advancement": group_advancement_counter,
        "playoff_qualification": playoff_qualification_counter,
        "champion_given_playoff": champion_given_playoff_counter,
        "simulations": simulations,
    }

In [25]:
def merge_simulation_results(results_list):
    counter_keys = [
        "champion",
        "runner_up",
        "third_place",
        "fourth_place",
        "final",
        "semifinal",
        "quarterfinal",
        "round_of_16",
        "round_of_32",
        "group_advancement",
        "playoff_qualification",
        "champion_given_playoff",
    ]

    merged_results = {
        key: Counter()
        for key in counter_keys
    }

    total_simulations = 0

    for result in results_list:
        total_simulations += int(result["simulations"])

        for key in counter_keys:
            if key in result:
                merged_results[key].update(result[key])

    merged_results["simulations"] = total_simulations

    return merged_results

In [26]:
def run_simulation_chunk(
    chunk_id,
    simulations_in_chunk,
    base_seed,
    schedule_data,
    playoff_data,
    initial_states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
):
    path_a.FEATURE_COLUMNS = FEATURE_COLUMNS_USED

    # Se você estiver usando o ajuste de mercado via odds, mantenha esta linha.
    # Se não estiver usando o blend com odds, pode comentar.
    path_a.simulate_match = simulate_match_with_market_blend

    chunk_seed = int(base_seed + 1000003 * chunk_id)

    result = run_simulations_official_bracket(
        schedule_data=schedule_data,
        playoff_data=playoff_data,
        initial_states=initial_states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        simulations=simulations_in_chunk,
        seed=chunk_seed,
    )

    return result

In [27]:
TOTAL_SIMULATIONS = 100_000
BASE_SEED = 42

N_JOBS = max(1, os.cpu_count() - 1)
N_CHUNKS = N_JOBS * 4

chunk_sizes = split_simulations(
    total_simulations=TOTAL_SIMULATIONS,
    n_chunks=N_CHUNKS,
)

print("CPUs disponíveis:", os.cpu_count())
print("N_JOBS:", N_JOBS)
print("N_CHUNKS:", len(chunk_sizes))
print("Primeiros tamanhos dos blocos:", chunk_sizes[:10])
print("Total:", sum(chunk_sizes))

CPUs disponíveis: 12
N_JOBS: 11
N_CHUNKS: 44
Primeiros tamanhos dos blocos: [2273, 2273, 2273, 2273, 2273, 2273, 2273, 2273, 2273, 2273]
Total: 100000


In [28]:
start_time = time.time()

parallel_results = Parallel(
    n_jobs=N_JOBS,
    backend="loky",
    verbose=10,
)(
    delayed(run_simulation_chunk)(
        chunk_id=chunk_id,
        simulations_in_chunk=chunk_size,
        base_seed=BASE_SEED,
        schedule_data=schedule_data,
        playoff_data=None,
        initial_states=initial_states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
    )
    for chunk_id, chunk_size in enumerate(chunk_sizes)
)

simulation_results = merge_simulation_results(parallel_results)

elapsed_time = time.time() - start_time

print("Simulações finais:", simulation_results["simulations"])
print("Tempo total em segundos:", round(elapsed_time, 2))
print("Tempo total em minutos:", round(elapsed_time / 60, 2))

[Parallel(n_jobs=11)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=11)]: Done   3 tasks      | elapsed: 28.1min
[Parallel(n_jobs=11)]: Done  10 tasks      | elapsed: 32.7min
[Parallel(n_jobs=11)]: Done  19 tasks      | elapsed: 64.0min
[Parallel(n_jobs=11)]: Done  28 out of  44 | elapsed: 88.9min remaining: 50.8min
[Parallel(n_jobs=11)]: Done  33 out of  44 | elapsed: 100.3min remaining: 33.4min
[Parallel(n_jobs=11)]: Done  38 out of  44 | elapsed: 116.1min remaining: 18.3min


Simulações finais: 100000
Tempo total em segundos: 7395.15
Tempo total em minutos: 123.25


[Parallel(n_jobs=11)]: Done  44 out of  44 | elapsed: 123.3min finished


In [29]:
simulation_results

{'champion': Counter({'Spain': 21405,
          'Argentina': 10121,
          'France': 9064,
          'Brazil': 7606,
          'Germany': 7545,
          'England': 5261,
          'Netherlands': 4139,
          'Colombia': 3890,
          'Belgium': 3597,
          'Portugal': 3359,
          'Norway': 2884,
          'Switzerland': 2853,
          'Uruguay': 2211,
          'Japan': 2090,
          'Ecuador': 2013,
          'Mexico': 1974,
          'Morocco': 1402,
          'Austria': 956,
          'United States': 938,
          'Turkey': 896,
          'South Korea': 857,
          'Canada': 644,
          'Paraguay': 611,
          'Croatia': 595,
          'Scotland': 577,
          'Senegal': 452,
          'Sweden': 399,
          'Algeria': 325,
          'Iran': 294,
          'Australia': 243,
          'Egypt': 165,
          'Ivory Coast': 164,
          'Panama': 82,
          'Uzbekistan': 78,
          'Tunisia': 67,
          'Jordan': 47,
          'Saudi Arabi

In [30]:
total = simulation_results["simulations"]

checks = {
    "champion": 1 * total,
    "runner_up": 1 * total,
    "third_place": 1 * total,
    "fourth_place": 1 * total,
    "final": 2 * total,
    "semifinal": 4 * total,
    "quarterfinal": 8 * total,
    "round_of_16": 16 * total,
    "round_of_32": 32 * total,
    "group_advancement": 32 * total,
}

for key, expected_total in checks.items():
    observed_total = sum(simulation_results[key].values())

    print(
        key,
        "observado:",
        observed_total,
        "esperado:",
        expected_total,
        "ok:",
        observed_total == expected_total,
    )

champion observado: 100000 esperado: 100000 ok: True
runner_up observado: 100000 esperado: 100000 ok: True
third_place observado: 100000 esperado: 100000 ok: True
fourth_place observado: 100000 esperado: 100000 ok: True
final observado: 200000 esperado: 200000 ok: True
semifinal observado: 400000 esperado: 400000 ok: True
quarterfinal observado: 800000 esperado: 800000 ok: True
round_of_16 observado: 1600000 esperado: 1600000 ok: True
round_of_32 observado: 3200000 esperado: 3200000 ok: True
group_advancement observado: 3200000 esperado: 3200000 ok: True


In [31]:
def save_conditional_playoff_champion_probabilities_safe(
    champion_given_playoff_counter,
    playoff_qualification_counter,
    output_path,
):
    columns = [
        "team",
        "playoff_qualification_count",
        "champion_count",
        "champion_probability_given_qualification",
    ]

    if not playoff_qualification_counter:
        pd.DataFrame(columns=columns).to_csv(output_path, index=False)
        return

    rows = []

    for team, qualification_count in playoff_qualification_counter.items():
        champion_count = champion_given_playoff_counter.get(team, 0)

        rows.append(
            {
                "team": team,
                "playoff_qualification_count": qualification_count,
                "champion_count": champion_count,
                "champion_probability_given_qualification": (
                    champion_count / qualification_count
                    if qualification_count > 0
                    else 0.0
                ),
            }
        )

    probabilities_data = pd.DataFrame(rows, columns=columns).sort_values(
        "champion_probability_given_qualification",
        ascending=False,
    )

    probabilities_data.to_csv(output_path, index=False)


path_a.save_conditional_playoff_champion_probabilities = (
    save_conditional_playoff_champion_probabilities_safe
)

In [32]:
def save_counter_probabilities_safe(
    counter,
    simulations,
    output_path,
    probability_column,
):
    rows = [
        {
            "team": team,
            "count": count,
            probability_column: count / simulations,
        }
        for team, count in counter.most_common()
    ]

    probabilities_data = pd.DataFrame(
        rows,
        columns=["team", "count", probability_column],
    )

    probabilities_data.to_csv(output_path, index=False)


def save_outputs_with_round_tracking(results, output_dir, seed):
    output_dir.mkdir(parents=True, exist_ok=True)

    simulations = int(results["simulations"])

    output_specs = {
        "champion": ("champion_probabilities.csv", "champion_probability"),
        "runner_up": ("runner_up_probabilities.csv", "runner_up_probability"),
        "third_place": ("third_place_probabilities.csv", "third_place_probability"),
        "fourth_place": ("fourth_place_probabilities.csv", "fourth_place_probability"),
        "final": ("final_probabilities.csv", "final_probability"),
        "semifinal": ("semifinal_probabilities.csv", "semifinal_probability"),
        "quarterfinal": ("quarterfinal_probabilities.csv", "quarterfinal_probability"),
        "round_of_16": ("round_of_16_probabilities.csv", "round_of_16_probability"),
        "round_of_32": ("round_of_32_probabilities.csv", "round_of_32_probability"),
        "group_advancement": (
            "group_advancement_probabilities.csv",
            "group_advancement_probability",
        ),
        "playoff_qualification": (
            "playoff_qualification_probabilities.csv",
            "playoff_qualification_probability",
        ),
    }

    for result_key, (filename, probability_column) in output_specs.items():
        save_counter_probabilities_safe(
            counter=results[result_key],
            simulations=simulations,
            output_path=output_dir / filename,
            probability_column=probability_column,
        )

    metadata = {
        "model": "path_a_poisson_dynamic_ratings_with_round_tracking",
        "simulations": simulations,
        "seed": seed,
        "rounds_saved": list(output_specs.keys()),
    }

    (output_dir / "run_metadata.json").write_text(
        path_a.json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

In [33]:
save_outputs_with_round_tracking(
    results=simulation_results,
    output_dir=output_dir,
    seed=42,
)

In [34]:
simulation_results

{'champion': Counter({'Spain': 21405,
          'Argentina': 10121,
          'France': 9064,
          'Brazil': 7606,
          'Germany': 7545,
          'England': 5261,
          'Netherlands': 4139,
          'Colombia': 3890,
          'Belgium': 3597,
          'Portugal': 3359,
          'Norway': 2884,
          'Switzerland': 2853,
          'Uruguay': 2211,
          'Japan': 2090,
          'Ecuador': 2013,
          'Mexico': 1974,
          'Morocco': 1402,
          'Austria': 956,
          'United States': 938,
          'Turkey': 896,
          'South Korea': 857,
          'Canada': 644,
          'Paraguay': 611,
          'Croatia': 595,
          'Scotland': 577,
          'Senegal': 452,
          'Sweden': 399,
          'Algeria': 325,
          'Iran': 294,
          'Australia': 243,
          'Egypt': 165,
          'Ivory Coast': 164,
          'Panama': 82,
          'Uzbekistan': 78,
          'Tunisia': 67,
          'Jordan': 47,
          'Saudi Arabi